# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 0. Repository map and end-to-end flow

`generate_data.py` → `data/raw/*.csv` → audit → clean → `data/clean/*.csv` + `data/instamart.db` → RFM/segments + lapse model → privacy-safe ticket triage → policy retrieval + refund tools.

Teaching point: **fetching data** does not always mean calling an API. Here, the source is a local CSV export. In a production system, the same ingestion layer could use an approved API or warehouse connector, but credentials and network access are intentionally out of scope.

In [27]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\Users\snith\OneDrive\Desktop\instamart-customer-intelligence-copilot\instamart_customer_copilot_FOLDER
Root files: ['.gitignore', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [28]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [29]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


1. Which columns are supposed to be unique keys?
- customers: customer_id – uniquely identifies each customer.
- products: product_id – uniquely identifies each product.
- orders: order_id – uniquely identifies each order.
- tickets: ticket_id – uniquely identifies each support ticket.
These columns should not contain duplicate values. customer_id and product_id in orders are foreign keys, so they can repeat.


2. Are duplicates exact duplicates or multiple states of the same order?
We need to distinguish between exact duplicate rows and repeated order_id values.
- Exact duplicates: The entire row is repeated, indicating possible accidental duplication.
- Repeated order IDs: The same order may appear multiple times because of different order states or updates.
We should investigate before deleting duplicates to avoid losing valid order information.

3. Which values are impossible?
We should identify and validate the following:
- Negative delivery duration: Delivery minutes cannot be negative.
- Non-numeric money: price_inr and order_amount must contain valid numeric values.
- Implausible age: Age should be numeric and within a reasonable range (e.g., 1–100).
- Invalid quantity: Order quantity should be a positive number.
- Invalid discount: discount_pct should be between 0 and 100.
These values can affect the accuracy of business analysis and machine learning.

4. Why should we distinguish an empty string from a missing value?
An empty string ("") means the field exists but contains no text, whereas a missing value (NaN/NULL) represents an absent value.For example, the customers dataset has 61 blank ages and 30 blank genders, while the missing count is zero.

If we treat them as the same without checking, we may underestimate missing data and apply incorrect cleaning strategies.
5. What would go wrong if we modelled before cleaning?
1)Incorrect predictions: Invalid values and duplicates can mislead the model.
2) Biased results: Missing values may affect customer behaviour analysis.
3) Poor model performance: Inconsistent data reduces model accuracy.
4)Wrong business insights: Duplicate orders can inflate revenue and order counts.
5) Unreliable decisions: Customer segmentation and churn predictions may become inaccurate.

In [38]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [39]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [41]:
import sqlite3
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    q2 = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)
display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [ ]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()

In [42]:
print("RFM shape:", rfm.shape)

print("\nRFM statistics:")
display(rfm[["recency", "frequency", "monetary"]].agg(
    ["mean", "median", "min", "max"]
))

print("\nSegment profile:")
display(segment_profile)

print("\nSegment sizes:")
display(rfm["segment_name"].value_counts())

print("\nPremium customers:")
print(rfm.groupby("segment_name")["monetary"].mean())

RFM shape: (1175, 7)

RFM statistics:


,recency,frequency,monetary
mean,33.558298,9.455319,2552.058553
median,19.000000,5.000000,2286.500000
min,0.000000,1.000000,12.000000
max,180.000000,33.000000,14766.550000



Segment profile:


,segment_id,size,recency,frequency,monetary
0,0,436,28.862385,3.772936,1057.618234
1,1,138,24.115942,6.768116,7129.695652
2,2,419,8.766110,19.403341,3410.020525
3,3,182,109.043956,2.203297,685.992582



Segment sizes:


segment_name
At Risk     618
Frequent    419
Premium     138
Name: count, dtype: int64


Premium customers:
segment_name
At Risk      948.175081
Frequent    3410.020525
Premium     7129.695652
Name: monetary, dtype: float64


In [45]:
%pip install imbalanced-learn
import imblearn
print("imbalanced-learn version:", imblearn.__version__)


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
imbalanced-learn version: 0.14.2


In [46]:
from imblearn.over_sampling import RandomOverSampler

print("RandomOverSampler imported successfully!")

RandomOverSampler imported successfully!


In [ ]:
print("Target distribution (full cohort):")
display(pd.Series(lapse_summary["class_distribution"], name="customers"))
print("Training distribution:")
display(pd.Series(lapse_summary["training_distribution"], name="customers"))
print("Test distribution (never oversampled):")
display(pd.Series(lapse_summary["testing_distribution"], name="customers"))

model_comparison = pd.DataFrame.from_dict(lapse_summary["models"], orient="index")
display(model_comparison[["accuracy", "precision", "recall", "f1", "confusion_matrix"]])
print("Selected model:", lapse_summary["selected_model"])
print(lapse_summary["feature_importance_explanation"])
display(lapse_summary["feature_importance"].head(10))
print("Premium eligibility uses historical RFM as of", lapse_summary["retention_eligibility_as_of"])
display(lapse_summary["retention_list"])

1. Which dates can enter the lapse-model features?

Only orders placed before the September prediction window can be used to calculate the model features. This means orders dated up to August 31. These historical orders provide the information available when making the prediction.

2. Why must September orders not enter the features?

September orders occur during the prediction period and represent future information. Including them would cause data leakage and make the model performance appear better than it would be in a real-world prediction made before September.

3. Why must Premium retention eligibility use only RFM values available before the prediction window?

Premium eligibility must be based only on information available at prediction time. Therefore, RFM values should be calculated using orders placed before September. Using later orders would introduce future information and lead to data leakage in the eligibility decision.

4. Why can accuracy be misleading when a substantial minority of customers lapse?

A model can achieve high accuracy by mostly predicting the majority class while failing to identify customers who are likely to lapse. Therefore, accuracy alone is not sufficient. We should also examine the confusion matrix, precision, recall, and F1-score for the lapse class to evaluate how effectively the model identifies customers who may stop ordering.

## 6. Part C — Support ticket triage with guardrails

The implementation remains in `src/triage.py`; the single Part C execution cell below calls `run_pipeline` and displays the results. Ticket order is preserved from `data/raw/tickets.csv`.

### Task 1 — Input guardrails
`run_pipeline` checks whitespace-only text and `mock_llm.est_tokens(text) > MAX_INPUT_TOKENS` (`400`) before calling the model. It assigns `skipped_empty` or `flagged_long`; the summary confirms these tickets were excluded from model calls.

### Task 2 — PII redaction
`PATTERNS` and `redact` apply the email, phone, and order-ID expressions in the required order. `run_pipeline` reports replacement counts, checks for residual matches, and asserts that each model user message is exactly the redacted ticket text.

### Task 3 — Zero-shot and few-shot prompts
`ZERO` is the JSON-only zero-shot system prompt. `FEW` adds an `EXAMPLES:` section with labelled examples, including fraud. The execution cell runs each prompt on the full eligible ticket set.

### Task 4 — Resilient model calls
`retry_chat` wraps every model call with at most three retries and exponential backoff plus jitter using `BASE_DELAY`. `run_pipeline` calls `mock_llm.reset()` at the beginning of each run. The retry summary below reports recovered retry attempts for each run.

### Task 5 — Output validation and human review
`_validate_response` accepts only an allowed JSON category, or exactly one allowed category word found by regex in non-JSON text. Invalid outputs are marked `flagged_invalid_output`, left without a category, and routed to `human_review`—never guessed.

### Task 6 — Zero-shot versus few-shot comparison
The execution cell displays classified count, accuracy, total tokens, tokens per classified ticket, and the zero-shot confusion matrix. In the verified run, zero-shot achieved 85.7% (126/147) at 8,932 tokens (60.76 per ticket); few-shot achieved 100% (147/147) at 16,576 tokens (112.76 per ticket). Few-shot costs 7,644 additional tokens (about 52 more per classified ticket) but corrected 21 zero-shot category errors. For this dataset the extra cost is worthwhile when avoiding misclassification—especially fraud misroutes—matters more than token spend; lower-risk queues may choose zero-shot with human-review controls.

### Task 7 — Token budget, segment routing, and saved outputs
`run_pipeline(token_budget=...)` checks the cumulative token use before each call and assigns `budget_exhausted` when the budget is reached. It reads `outputs/customer_segments.csv`; human-review cases take precedence, followed by fraud/security, premium customers/premium desk, and other valid tickets/support queue. The notebook runs the zero-shot 6,000-token demo and the final few-shot 100,000-token run. Outputs: `outputs/triage_budget_demo.csv` and `outputs/triage_results.csv`.

**Interpretation:** accepted JSON and keyword classifications are rule-validated against the allowed categories. Empty, long, invalid-output, and budget-exhausted tickets are sent to human review; the pipeline does not invent a category for them.

In [ ]:
from src.triage import run_pipeline, FEW, ZERO

zero_results, zero_summary = run_pipeline(prompt=ZERO, token_budget=None, output_name=None)
few_results, few_summary = run_pipeline(prompt=FEW, token_budget=None, output_name=None)
expected_statuses = {
    "classified_json": 135, "classified_keyword": 12,
    "flagged_invalid_output": 9, "skipped_empty": 4,
    "flagged_long": 4, "budget_exhausted": 0
}
expected_redactions = {"email": 55, "phone": 91, "order_id": 132, "total": 278}
expected_no_remaining_pii = {"email": 0, "phone": 0, "order_id": 0, "total": 0}
assert zero_summary["status_counts"] == expected_statuses
assert few_summary["status_counts"] == expected_statuses
assert zero_summary["redactions"] == expected_redactions
assert few_summary["redactions"] == expected_redactions
assert zero_summary["remaining_pii"] == expected_no_remaining_pii
assert few_summary["remaining_pii"] == expected_no_remaining_pii
assert zero_summary["retry_count"] == few_summary["retry_count"] == 19
assert abs(zero_summary["accuracy"] - 126 / 147) < 0.005
assert few_summary["accuracy"] == 1.0
prompt_comparison = pd.DataFrame([
    {"prompt": "zero-shot", "classified": zero_summary["classified_tickets"],
     "accuracy": zero_summary["accuracy"], "total_tokens": zero_summary["total_tokens"],
     "tokens_per_classified_ticket": zero_summary["tokens_per_classified_ticket"]},
    {"prompt": "few-shot", "classified": few_summary["classified_tickets"],
     "accuracy": few_summary["accuracy"], "total_tokens": few_summary["total_tokens"],
     "tokens_per_classified_ticket": few_summary["tokens_per_classified_ticket"]}
])
display(prompt_comparison.round(3))
retry_summary = pd.DataFrame([
    {"prompt": "zero-shot", "retries": zero_summary["retry_count"],
     "model_calls": zero_summary["model_call_tickets"],
     "guardrail_model_calls": zero_summary["guardrail_model_calls"]},
    {"prompt": "few-shot", "retries": few_summary["retry_count"],
     "model_calls": few_summary["model_call_tickets"],
     "guardrail_model_calls": few_summary["guardrail_model_calls"]}
])
print("Retry and guardrail-call summary:")
display(retry_summary)
print("Zero-shot status counts:", zero_summary["status_counts"])
print("Few-shot status counts:", few_summary["status_counts"])
redaction_summary = pd.DataFrame({
    "substitutions": few_summary["redactions"],
    "matches_remaining": few_summary["remaining_pii"]
})
print("PII redaction summary (both prompts process the same tickets):")
display(redaction_summary)
display(zero_summary["confusion_matrix"])

budget_results, budget_summary = run_pipeline(
    prompt=ZERO, token_budget=6000, output_name="triage_budget_demo.csv"
)
print("Budget demo tokens:", budget_summary["total_tokens"])
print("Budget-exhausted tickets:", budget_summary["status_counts"]["budget_exhausted"])
budget_status_summary = budget_results.status.value_counts().rename_axis("status").reset_index(name="tickets")
display(budget_status_summary)
assert budget_summary["status_counts"]["budget_exhausted"] > 0

triage_results, triage_summary = run_pipeline(
    prompt=FEW, token_budget=100000, output_name="triage_results.csv"
)
print("Final few-shot run summary:", {
    key: triage_summary[key] for key in (
        "classified_tickets", "accuracy", "total_tokens",
        "tokens_per_classified_ticket", "redactions", "remaining_pii",
        "status_counts", "retry_count"
    )
})
display(triage_results.head(10))
final_status_summary = triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets")
print("Final status counts:")
display(final_status_summary)
final_routing_summary = triage_results.groupby(["route", "priority"], dropna=False).size().reset_index(name="tickets")
print("Final routing and priority summary:")
display(final_routing_summary)
print("Saved budget demo: outputs/triage_budget_demo.csv")
print("Saved final few-shot results: outputs/triage_results.csv")

 PART C DISCUSSION: 

 
1. Why is redaction important?

Redaction removes email addresses, phone numbers, and order IDs before sending ticket text to the model. It reduces privacy risks, but other sensitive information may remain. Therefore, data minimization and human oversight are also important.



2. Why are invalid categories sent to human review instead of guessed?

Invalid outputs may cause incorrect classifications and routing. Therefore, the pipeline flags invalid responses and sends them to human review instead of guessing.



3. Why should fraud override ordinary customer-segment routing?

Fraud cases may involve unauthorized transactions or account compromise. They must be handled by the security team with HIGH priority, regardless of the customer's premium status.



4. Why must empty and long tickets be blocked before model calls?

Empty tickets have no useful information, while long tickets exceed the 400-token input limit. Blocking them avoids unnecessary model calls and token costs. They are sent for human review.

## 7. Part D policy retrieval and grounded answers

`src/policy_rag.py` splits the five local policy documents into 40-word chunks with 10-word overlap, fits one `TfidfVectorizer`, and ranks chunks by cosine similarity. Term frequency (TF) measures how often a term appears in a chunk; inverse document frequency (IDF) reduces the weight of terms shared by many chunks. A lexical TF-IDF search can miss semantically equivalent phrasings that use different words.

`PolicyRAG.answer` sends the three retrieved chunks with their source labels and the question to the offline mock. It instructs the assistant to answer only from the context and to use the `I could not find` fallback when the context does not contain the answer. The code cell reports chunk counts, the 14-question evaluation, and a second evaluation with zero overlap to make chunk-boundary failures visible.

In [49]:
import importlib
import src.policy_rag
importlib.reload(src.policy_rag)
from src.policy_rag import PolicyRAG, evaluate, retrieval_explanation

rag = PolicyRAG(chunk_size=40, overlap=10)
chunk_counts = pd.Series([chunk["source"] for chunk in rag.chunks]).value_counts().rename_axis("source").reset_index(name="chunks")
print("Overlapping chunks:", len(rag.chunks))
display(chunk_counts)
assert len(rag.chunks) == 15
assert dict(zip(chunk_counts.source, chunk_counts.chunks)) == {
    "refund_policy.txt": 4, "delivery_policy.txt": 3,
    "escalation_policy.txt": 3, "payment_policy.txt": 3,
    "privacy_policy.txt": 2,
}
print(retrieval_explanation())
refund_query_sources = [hit["source"] for hit in rag.retrieve("refund policy late delivery", k=3)]
print("Top sources for 'refund policy late delivery':", refund_query_sources)
assert refund_query_sources == ["refund_policy.txt"] * 3
print("Grounded example:")
display(rag.answer("What delivery time makes a completed order eligible for a full refund?"))

policy_eval = evaluate(rag)
no_overlap_rag = PolicyRAG(chunk_size=40, overlap=0)
no_overlap_eval = evaluate(no_overlap_rag)

def retrieval_scorecard(results, chunking):
    answerable = results[results.expected_doc != "NONE"]
    unanswerable = results[results.expected_doc == "NONE"]
    return {
        "chunking": chunking,
        "expected document in top 3": f"{int(answerable.retrieved_doc.sum())}/{len(answerable)}",
        "expected fact found": f"{int(answerable.fact_found.sum())}/{len(answerable)}",
        "unanswerable fallback": f"{int(unanswerable.fallback.sum())}/{len(unanswerable)}",
    }

scorecard = pd.DataFrame([
    retrieval_scorecard(policy_eval, "40 words, 10-word overlap"),
    retrieval_scorecard(no_overlap_eval, "40 words, no overlap"),
])
display(scorecard)
zero_answerable = no_overlap_eval[no_overlap_eval.expected_doc != "NONE"]
zero_unanswerable = no_overlap_eval[no_overlap_eval.expected_doc == "NONE"]
assert int(zero_answerable.retrieved_doc.sum()) == 12
assert int(zero_answerable.fact_found.sum()) == 10
assert int(zero_unanswerable.fallback.sum()) == 2
print("Answerable questions affected by zero overlap:")
boundary_failures = no_overlap_eval.loc[
    (no_overlap_eval.expected_doc != "NONE") & ~no_overlap_eval.fact_found,
    ["question", "expected_fact", "answer", "sources"],
]
display(boundary_failures)
assert int(policy_eval.loc[policy_eval.expected_doc != "NONE", "retrieved_doc"].sum()) == 12
assert int(policy_eval.loc[policy_eval.expected_doc != "NONE", "fact_found"].sum()) == 12
assert int(policy_eval.loc[policy_eval.expected_doc == "NONE", "fallback"].sum()) == 2
assert len(boundary_failures) >= 1

Overlapping chunks: 15


,source,chunks
0,refund_policy.txt,4
1,delivery_policy.txt,3
2,escalation_policy.txt,3
3,payment_policy.txt,3
4,privacy_policy.txt,2


Term frequency (TF) measures how often a term appears in a chunk, while inverse document frequency (IDF) gives less weight to terms common across many chunks and more weight to distinctive terms. TF-IDF is lexical, so different phrasings with the same meaning may have little or no overlap and retrieve the wrong passage.
Top sources for 'refund policy late delivery': ['refund_policy.txt', 'refund_policy.txt', 'refund_policy.txt']
Grounded example:


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.',
 'sources': ['refund_policy.txt', 'privacy_policy.txt'],
 'retrieved': [{'source': 'refund_policy.txt',
   'score': 0.35464416504570484,
   'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'},
  {'source': 'refund_policy.txt',
   'score': 0.31425507733397284,
   'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'},
  {'source': 'privacy_policy.txt',
   'score': 0.15759666156595936,
   'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account delet

,chunking,expected document in top 3,expected fact found,unanswerable fallback
0,"40 words, 10-word overlap",12/12,12/12,2/2
1,"40 words, no overlap",12/12,10/12,2/2


Answerable questions affected by zero overlap:


,question,expected_fact,answer,sources
0,What delivery time makes a completed order eli...,more than 45 minutes,receives a full refund of the order amount.,"[refund_policy.txt, privacy_policy.txt]"
7,How soon is money returned when a payment fail...,48 hours,"If a payment fails but money is deducted, the ...","[payment_policy.txt, refund_policy.txt, privac..."


8. Part D database-backed refund agent

The five JSON-described tools in `src/refund_agent.py` look up orders, retrieve policy, calculate eligibility, issue eligible refunds, or record human escalations. Refund tools independently re-check completed-order status, delivery over 45 minutes, and the INR 1000 automatic-approval ceiling before writing the JSONL ledger.

The ReAct loop passes every mock reply through the tool map and enforces model-call, token, and elapsed-time limits. Each tool observation is checkpointed under `state/`; `resume_agent` continues from that saved message history without repeating completed tools. The code cell runs all seven scenarios, records their results, demonstrates token/time fallbacks, and simulates a crash after the second S1 tool call.

In [53]:
from pathlib import Path
import json
import pandas as pd
import importlib
import src.refund_agent
importlib.reload(src.refund_agent)
from src.refund_agent import TOOLS_SPEC, TOOL_MAP, run_agent, resume_agent

root = Path.cwd()
while not (root / "data" / "raw" / "agent_scenarios.csv").exists() and root != root.parent:
    root = root.parent
scenario_path = root / "data" / "raw" / "agent_scenarios.csv"
scenarios = pd.read_csv(scenario_path, keep_default_na=False)
assert len(scenarios) == 7
assert len(TOOLS_SPEC) == 5 and set(TOOL_MAP) == {tool["name"] for tool in TOOLS_SPEC}

scenario_rows = []
run_records = []
for scenario in scenarios.itertuples(index=False):
    result = run_agent(scenario.user_message, scenario_id=scenario.scenario_id)
    expected_refund = float(scenario.expected_refund_inr) if scenario.expected_refund_inr else None
    action_tools = {"issue_refund", "escalate_to_human"}
    action = next((step for step in result["steps"] if step["tool"] in action_tools), None)
    refund_amount = (
        float(action["observation"]["refund_inr"])
        if action
        else (0.0 if expected_refund is not None else None)
    )
    assert result["status"] == scenario.expected_status
    if expected_refund is not None:
        assert refund_amount is not None and abs(refund_amount - expected_refund) <= 0.01
    if scenario.scenario_id == "S1":
        assert result["llm_calls"] == 5
    if scenario.scenario_id == "S7":
        assert result["status"] == "FALLBACK_MAX_ITERATIONS" and result["llm_calls"] == 6
        assert result["tokens_used"] <= 6000
    scenario_rows.append({
        "scenario_id": scenario.scenario_id,
        "status": result["status"],
        "refund_amount_inr": refund_amount,
        "model_calls": result["llm_calls"],
        "tokens_used": result["tokens_used"],
    })
    run_records.append({
        "scenario_id": scenario.scenario_id,
        "status": result["status"],
        "final_answer": result["final_answer"],
        "llm_calls": result["llm_calls"],
        "tokens_used": result["tokens_used"],
        "steps": result["steps"],
    })

(root / "outputs").mkdir(exist_ok=True)
run_log_path = root / "outputs" / "agent_runs.jsonl"
run_log_path.write_text("", encoding="utf-8")
with run_log_path.open("a", encoding="utf-8") as run_log:
    for record in run_records:
        run_log.write(json.dumps(record, ensure_ascii=False) + "\n")
scenario_summary = pd.DataFrame(scenario_rows)
display(scenario_summary)

s1_message = scenarios.loc[scenarios.scenario_id == "S1", "user_message"].iloc[0]
token_demo = run_agent(s1_message, scenario_id="S1-token-limit-demo", token_limit=200)
timeout_demo = run_agent(s1_message, scenario_id="S1-timeout-demo", time_limit_seconds=0)
assert token_demo["status"] == "FALLBACK_TOKEN_LIMIT"
assert [step["tool"] for step in token_demo["steps"]] == ["get_order", "search_policy"]
assert timeout_demo["status"] == "FALLBACK_TIMEOUT" and timeout_demo["llm_calls"] == 0
print("S1 token-limit demo:", token_demo["status"], "tokens:", token_demo["tokens_used"])
print("S1 timeout demo:", timeout_demo["status"], "model calls:", timeout_demo["llm_calls"])

crash_run_id = "crash-resume-S1"
try:
    run_agent(s1_message, scenario_id="S1-crash-resume", run_id=crash_run_id, crash_after=2)
except RuntimeError as error:
    assert str(error) == "Simulated crash after checkpoint was saved"
else:
    raise AssertionError("Expected the simulated crash after the second tool")
checkpoint = json.loads((root / "state" / f"{crash_run_id}.json").read_text(encoding="utf-8"))
assert checkpoint["tools_executed"] == ["get_order", "search_policy"]
resumed = resume_agent(crash_run_id)
crash_resume_tools = [step["tool"] for step in resumed["steps"]]
assert crash_resume_tools == ["get_order", "search_policy", "calculate_refund", "issue_refund"]
assert resumed["status"] == "REFUND" and abs(resumed["steps"][-1]["observation"]["refund_inr"] - 128.25) <= 0.01
print("Crash/resume tools:", crash_resume_tools)
print("Crash/resume result:", resumed["status"], resumed["final_answer"])
print("Run log: outputs/agent_runs.jsonl")
print("Refund action ledger: outputs/refund_ledger.jsonl")

,scenario_id,status,refund_amount_inr,model_calls,tokens_used
0,S1,REFUND,128.25,5,887
1,S2,REFUND,135.00,5,887
2,S3,ESCALATE,1999.00,5,894
3,S4,DENIED,0.00,4,605
4,S5,DENIED,0.00,3,346
5,S6,NOT_FOUND,0.00,2,84
6,S7,FALLBACK_MAX_ITERATIONS,NaN,6,3096


S1 token-limit demo: FALLBACK_TOKEN_LIMIT tokens: 348
S1 timeout demo: FALLBACK_TIMEOUT model calls: 0
Crash/resume tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']
Crash/resume result: REFUND REFUND: INR 128.25 issued.
Run log: outputs/agent_runs.jsonl
Refund action ledger: outputs/refund_ledger.jsonl


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [54]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
(not created)
\noutputs
(not created)
\nstate
(not created)
Database exists: True
